# Fine-tune Surya OCR 2.5 with ms-swift (Colab)

The same training as `Surya_finetune_Colab.ipynb` (full fine-tune, vision encoder frozen, its merger and the
language model trained, 8-bit AdamW, float32 on a T4), run by [ms-swift](https://github.com/modelscope/ms-swift)
4.6.0 instead of a hand-written loop.

ms-swift knows Qwen3.5 (`model_type` and `template` `qwen3_5`), but Surya is a Qwen3.5 with **its own vocabulary**,
so three things in ms-swift are set right for Surya first (cell 2); cell 4 then checks that one page is encoded
exactly as the hand-written notebook encodes it, and stops if not:

1. **tokenizer**: transformers loads a Qwen3.5 tokenizer for Surya, which splits Arabic into byte pieces
   (61 tokens instead of 37 for one line); here Surya's own (`TokenizersBackend`) is always loaded;
2. **image token**: ms-swift's Qwen3.5 template has Qwen's id for `<|image_pad|>` (248056); Surya's is 11;
3. **end token**: ms-swift joins the answer and `<|im_end|>` into one text before tokenizing; Surya's vocabulary
   has tokens such as `.<` that then swallow the `<` of `<|im_end|>`, so the model would not learn to stop.
   Here the end token is added as its id (2).

Also as in the other notebook: no `<think>` prefix (Surya's answers have none), images resized within Surya's
limits, picture-description words left out of the loss.

**Run:** *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. The batch folders are the same as for
the other notebook (`MyDrive/surya/batches/epoch_001`, ...). Checkpoints go to `MyDrive/surya/run_swift/`;
after a disconnect, *Run all* continues from the last one.

In [ ]:
%pip install -q "ms-swift==4.6.0" "transformers==5.19.0" "bitsandbytes==0.50.2" "qwen_vl_utils==0.0.14" decord
# ms-swift 4.6.0 needs transformers below 5.20 (5.19.0 has Qwen3.5 and Surya's tokenizer class).
# If Colab says to restart: Runtime > Restart session, then Run all again.

In [ ]:
import json, os, re, time
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less GPU memory lost to fragmentation
os.environ["USE_HF"] = "1"                                          # the model from Hugging Face, not ModelScope
# images resized within Surya's limits (at least 1792x28, at most 3072x2048 pixels), counted in 32x32 pieces
os.environ["IMAGE_MIN_TOKEN_NUM"] = str(1792 * 28 // 32**2)   # 49
os.environ["IMAGE_MAX_TOKEN_NUM"] = str(3072 * 2048 // 32**2)  # 6144

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

MODEL = "datalab-to/surya-ocr-2.5"
DATA = Path("/content/drive/MyDrive/surya/batches")   # epoch_001/, epoch_002/, ...
EPOCHS = [1, 2]            # batch folders to train on, one after the other (each page once)
RUN = Path("/content/drive/MyDrive/surya/run_swift")   # checkpoints; the trained model at the end
LR = 1e-5
GRAD_ACCUM = 8
MAX_LENGTH = 6144          # image + prompt + answer tokens; longer pages are left out
SAVE_EVERY = 100           # optimizer steps between checkpoints (and validation loss)
VAL_PAGES = 32             # validation pages for the loss
RUN.mkdir(parents=True, exist_ok=True)

import torch
from transformers import AutoConfig, AutoProcessor, AutoTokenizer, TokenizersBackend

# 1. tokenizer: Surya's own, wherever ms-swift (or anything else) loads a tokenizer or processor
def surya_tokenizer(path, **kwargs):
    return TokenizersBackend.from_pretrained(path, **{k: v for k, v in kwargs.items()
                                                      if k in ("revision", "cache_dir", "token", "local_files_only")})

if not getattr(AutoProcessor, "surya_patched", False):
    _processor_from_pretrained = AutoProcessor.from_pretrained.__func__
    def _processor(cls, path, *args, **kwargs):
        kwargs.setdefault("tokenizer", surya_tokenizer(path, **kwargs))
        return _processor_from_pretrained(cls, path, *args, **kwargs)
    AutoProcessor.from_pretrained = classmethod(_processor)
    AutoTokenizer.from_pretrained = classmethod(lambda cls, path, *args, **kwargs: surya_tokenizer(path, **kwargs))
    AutoProcessor.surya_patched = True

from swift.template import TEMPLATE_MAPPING
from swift.template.templates.qwen import Qwen3_5Template

# 2. image and video tokens: Surya's ids
config = AutoConfig.from_pretrained(MODEL)
Qwen3_5Template.image_token_id, Qwen3_5Template.video_token_id = config.image_token_id, config.video_token_id
# 3. end token: by id, so it is never tokenized together with the answer's last characters
TEMPLATE_MAPPING["qwen3_5"].suffix = [["eos_token_id"]]

# picture descriptions: ours are one stock sentence per kind of picture, Surya describes each picture; their words
# are not learned (the tags around them are)
DESCRIPTION = re.compile(r'<img alt="Photo: ([^"]*)"/><p class="img-desc">([^<]*)</p>')


def mask_descriptions(labels, target, tokenizer):
    """labels with -100 on answer tokens at least half inside a picture description. False if the answer's tokens
    are not where expected (then nothing is changed)."""
    spans = [m.span(g) for m in DESCRIPTION.finditer(target) for g in (1, 2)]
    if not spans:
        return True
    answer = tokenizer(target, add_special_tokens=False, return_offsets_mapping=True)
    ids = answer["input_ids"]
    start = next((i for i, t in enumerate(labels) if t != -100), None)
    if start is None or labels[start:start + len(ids)] != ids:
        return False
    for k, (s, e) in enumerate(answer["offset_mapping"]):
        if 2 * sum(max(0, min(e, b) - max(s, a)) for a, b in spans) >= e - s > 0:
            labels[start + k] = -100
    return True


if not getattr(Qwen3_5Template, "surya_patched", False):
    _encode = Qwen3_5Template._encode
    def _encode_masked(self, inputs):
        encoded = _encode(self, inputs)
        last = inputs.messages[-1] if inputs.messages else {}
        if encoded.get("labels") is not None and last.get("role") == "assistant" and isinstance(last.get("content"), str):
            if not mask_descriptions(encoded["labels"], last["content"], self.tokenizer):
                print("warning: picture descriptions not masked on one page (answer tokens not found)")
        return encoded
    Qwen3_5Template._encode = _encode_masked
    Qwen3_5Template.surya_patched = True
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU",
      "| image token", config.image_token_id, "| end token", config.eos_token_id)

## The data, in ms-swift's form

One line per page: the image, Surya's prompt as the user turn (`<image>` marks where the image goes), the target as
the answer. Train pages from every folder in `EPOCHS`; up to `VAL_PAGES` of their validation pages (never trained on) for
the validation loss.

In [ ]:
SWIFT_DATA = Path("/content/swift_data")
SWIFT_DATA.mkdir(exist_ok=True)


def pages(epoch):
    folder = DATA / f"epoch_{epoch:03d}"
    rows = [json.loads(line) for line in (folder / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if line]
    return [dict(row, folder=folder) for row in rows if row["task"] in ("full_ocr", "block_ocr")]


def swift_row(row):
    prompt = (row["folder"] / row["prompts"]).read_text(encoding="utf-8")
    target = (row["folder"] / row["targets"]).read_text(encoding="utf-8")
    return {"messages": [{"role": "user", "content": "<image>" + prompt}, {"role": "assistant", "content": target}],
            "images": [str(row["folder"] / row["images"])]}


train = [swift_row(r) for e in EPOCHS for r in pages(e) if r["split"] == "train"]
val = [swift_row(r) for e in EPOCHS for r in pages(e) if r["split"] == "val"][:VAL_PAGES]  # never trained on
for name, rows in (("train", train), ("val", val)):
    with open(SWIFT_DATA / f"{name}.jsonl", "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
print(len(train), "train pages,", len(val), "validation pages" + ("" if val else " (no validation loss)"))

## Check: one page encoded by ms-swift exactly as Surya is trained

ms-swift's encoding of one page (with a picture, so the description masking is checked too) against the
encoding of `Surya_finetune_Colab.ipynb`: Surya's processor and chat template for the prompt, the target's own
tokens, then the end token. Token ids, loss labels and image grid must all be the same; if not, this cell stops.

In [ ]:
from swift import get_processor, get_template

swift_processor = get_processor(MODEL, model_type="qwen3_5", use_hf=True)
template = get_template(swift_processor, template_type="qwen3_5", add_non_thinking_prefix=False)
template.set_mode("train")

sample = next((r for r in train if "img-desc" in r["messages"][1]["content"]), train[0])
encoded = template.encode(sample)

# the reference: as the hand-written notebook builds it
from PIL import Image
processor = AutoProcessor.from_pretrained(MODEL)  # Surya's tokenizer (patched above)
processor.image_processor.size = {"shortest_edge": 1792 * 28, "longest_edge": 3072 * 2048}
prompt, target = sample["messages"][0]["content"][len("<image>"):], sample["messages"][1]["content"]
image = Image.open(sample["images"][0]).convert("RGB")
reference = processor.apply_chat_template(
    [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}],
    add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
answer = processor.tokenizer(target, add_special_tokens=False)["input_ids"]
eos = processor.tokenizer.eos_token_id
ref_ids = reference["input_ids"][0].tolist() + answer + [eos]
ref_labels = [-100] * reference["input_ids"].shape[1] + answer + [eos]
assert mask_descriptions(ref_labels, target, processor.tokenizer)

got_ids, got_labels = list(encoded["input_ids"]), list(encoded["labels"])
checks = {"token ids": got_ids == ref_ids,
          "loss labels": got_labels == ref_labels,
          "image grid": encoded["image_grid_thw"].tolist() == reference["image_grid_thw"].tolist(),
          "ends with the end token": got_ids[-1] == eos and got_labels[-1] == eos}
for name, ok in checks.items():
    print(f"{'OK  ' if ok else 'DIFF'} {name}")
print(f"{len(got_ids)} tokens ({len(ref_ids)} expected), {sum(t != -100 for t in got_labels)} in the loss "
      f"({sum(t != -100 for t in ref_labels)} expected), {sum(t == -100 for t in ref_labels[-len(answer) - 1:])} "
      "picture-description tokens left out")
if not all(checks.values()):
    first = next((i for i, (a, b) in enumerate(zip(got_ids, ref_ids)) if a != b), min(len(got_ids), len(ref_ids)))
    tok = processor.tokenizer
    print("first difference at token", first)
    print("  ms-swift :", repr(tok.decode(got_ids[max(0, first - 8):first + 12])))
    print("  expected :", repr(tok.decode(ref_ids[max(0, first - 8):first + 12])))
    raise RuntimeError("ms-swift does not encode this page as Surya is trained; do not train. Send the output above.")
print("ms-swift encodes the page exactly as Surya is trained")

## Train

Full fine-tune: the vision encoder frozen (`freeze_vit`), its merger (`freeze_aligner=False`) and the language
model trained; float32 and 8-bit AdamW as in the other notebook. A checkpoint (and the validation loss) every
`SAVE_EVERY` steps; the last two are kept. After a disconnect, *Run all* continues from the newest.

In [ ]:
from swift import SftArguments, sft_main

checkpoints = sorted(RUN.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
args = SftArguments(
    model=MODEL, model_type="qwen3_5", template="qwen3_5", use_hf=True,
    tuner_type="full", torch_dtype="float32", attn_impl="sdpa",
    freeze_vit=True, freeze_aligner=False,
    dataset=[str(SWIFT_DATA / "train.jsonl")], val_dataset=[str(SWIFT_DATA / "val.jsonl")] if val else [],
    split_dataset_ratio=0.0,
    max_length=MAX_LENGTH, truncation_strategy="delete", add_non_thinking_prefix=False, lazy_tokenize=True,
    per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, num_train_epochs=1, lr_scheduler_type="cosine", warmup_ratio=0.05, weight_decay=0.0,
    max_grad_norm=1.0, optim="adamw_bnb_8bit", gradient_checkpointing=True,
    logging_steps=10, eval_strategy="steps" if val else "no", eval_steps=SAVE_EVERY, save_steps=SAVE_EVERY, save_total_limit=2,
    output_dir=str(RUN), add_version=False, report_to=["none"], dataloader_num_workers=2, seed=42,
    resume_from_checkpoint=str(checkpoints[-1]) if checkpoints else None)
result = sft_main(args)
print(result)

## The trained model

The last checkpoint is a complete model (weights, config, processor). It is copied to `MyDrive/surya/run_swift/model`
with Surya's tokenizer files, then read one validation page (or the first page) as a quick check. For the full comparison with the
original Surya, use the evaluation cells with `TRAINED = Path("/content/drive/MyDrive/surya/run_swift/model")`.

In [ ]:
import shutil
from transformers import Qwen3_5ForConditionalGeneration

last = Path(result.get("last_model_checkpoint") or sorted(RUN.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))[-1])
final = RUN / "model"
if final.exists():
    shutil.rmtree(final)
shutil.copytree(last, final, ignore=shutil.ignore_patterns("optimizer*", "scheduler*", "rng_state*", "global_step*", "trainer_state.json"))
TokenizersBackend.from_pretrained(MODEL).save_pretrained(final)  # Surya's own tokenizer files
print("model saved to", final, "from", last.name)

model = Qwen3_5ForConditionalGeneration.from_pretrained(str(final), dtype=torch.float32).to("cuda").eval()
page = (val or train)[0]
prompt, target = page["messages"][0]["content"][len("<image>"):], page["messages"][1]["content"]
inputs = processor.apply_chat_template(
    [{"role": "user", "content": [{"type": "image", "image": Image.open(page["images"][0]).convert("RGB")},
                                  {"type": "text", "text": prompt}]}],
    add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=4096, do_sample=False, use_cache=True)
new = out[0, inputs["input_ids"].shape[1]:]
answer = processor.decode(new, skip_special_tokens=True)
import difflib
print("stopped by itself:", bool((new == processor.tokenizer.eos_token_id).any()),
      f"| {difflib.SequenceMatcher(None, answer, target).ratio():.1%} similar to the target")
print(answer[:1500])